Step 1 : 환경 설정

In [1]:
# PySpark 및 파일 다운로드를 위한 wget 설치
!pip install pyspark wget

  Preparing metadata (setup.py) ... done
  Created wheel for wget: filename=wget-3.2-py3-none-any.whl size=9686 sha256=dbb7f90ce646731d0bc821a1bd6c604d2be33d286c128380eecc08b2112a2925
  Stored in directory: /root/.cache/pip/wheels/8a/b8/04/0c88fb22489b0c049bee4e977c5689c7fe597d6c4b0e7d0b6a
Successfully built wget


Step 2 : Spark 초기화

In [2]:
import wget
from pyspark.sql import SparkSession

# SparkSession 및 SparkContext 초기화
spark = SparkSession.builder.master("local[*]").appName("HW1_Q1").getOrCreate()
sc = spark.sparkContext

Step 3 : 데이터 로드

In [3]:
# 과제 제공 URL에서 텍스트 파일 다운로드
url = "https://raw.githubusercontent.com/nongaussian/class-2024-datamining/main/quiz/sherlock.txt"
wget.download(url, "sherlock.txt")

# 텍스트 파일을 읽어 RDD 생성
rdd = sc.textFile("sherlock.txt")

Step 4 : 데이터 처리 파이프라인

In [4]:
# Transformation 및 Action 연산 체이닝
q1_result = (
    rdd.flatMap(lambda line: line.split(" "))       # 공백 기준으로 단어 분리 (문장부호 유지)
       .filter(lambda word: len(word) > 2)          # 길이가 2보다 큰 단어만 통과
       .map(lambda word: word.lower()[:2])          # 대소문자 구분 없이 앞 두 글자(2-head)만 추출
       .map(lambda head: (head, 1))                 # Reduce를 위해 (키, 1) 쌍으로 변환
       .reduceByKey(lambda a, b: a + b)             # 동일한 키(2-head)의 빈도수 합산
       .filter(lambda pair: pair[1] < 32)           # 빈도수가 32 미만인 쌍만 통과
       .map(lambda pair: pair[1])                   # 값(빈도수)만 추출
       .sum()                                       # 추출된 빈도수들의 총합 계산 (Action)
)

Step 5 : 결과 확인 및 출력

In [5]:
# 최종 결과 출력
print("빈도가 32 미만인 2-head word들의 빈도 합:", q1_result)

빈도가 32 미만인 2-head word들의 빈도 합: 772
